# Basics &mdash; DeMorgan's Law and the Contrapositive Form

**Concept 7 of the Basics decomposition:** *DeMorgan's Law and the Contrapositive Form*

$\neg(a\wedge b)\equiv(\neg a\vee\neg b)$, and $(a\Rightarrow b)\equiv(\neg b\Rightarrow\neg a)$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-DeMorgan-And-Contrapositive/Concept-DeMorgan-And-Contrapositive.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Basics/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


**DeMorgan:**
$$\neg(a \wedge b) \equiv (\neg a \vee \neg b) \qquad
\neg(a \vee b) \equiv (\neg a \wedge \neg b)$$

Negation **swaps** $\wedge$ and $\vee$ and pushes inward. The same law holds for sets
($\overline{A\cap B} = \overline{A}\cup\overline{B}$), for languages
(Chapter 6, Concept 12) and for quantifiers (Concept 8).

**Contrapositive:**
$$(a \Rightarrow b) \equiv (\neg b \Rightarrow \neg a)$$

Not merely implied &mdash; **equivalent**. So proving one proves the other, and that is
the licence behind every "use the Pumping Lemma only to disprove" argument in
Chapter 4.

The near-miss to avoid: $(a\Rightarrow b)$ is **not** equivalent to
$(b\Rightarrow a)$, its *converse*, nor to $(\neg a\Rightarrow\neg b)$, its *inverse*.

## 2. Definitions

### The laws, as checkable functions

In [ ]:
IMP = lambda a, b: (not a) or b
BOOLS = [(a, b) for a in (False, True) for b in (False, True)]

def equiv2(f, g, rows=None):
    rows = rows or BOOLS
    return all(f(a, b) == g(a, b) for a, b in rows)

### The set and language versions

In [ ]:
def set_demorgan(U, A, B):
    return (U - (A & B)) == ((U - A) | (U - B)), (U - (A | B)) == ((U - A) & (U - B))

<!-- nav-strip -->

---

&larr;&nbsp;[Basics&nbsp;6.&nbsp;Logical Connectives and Predicates](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Logical-Connectives/Concept-Logical-Connectives.ipynb) &nbsp;&middot;&nbsp; [**Basics** index](https://github.com/ganeshutah/Jove/blob/master/Basics/README.md) &nbsp;&middot;&nbsp; [Basics&nbsp;8.&nbsp;Quantifiers as Repeated Conjunction and Disjunction, and Negating Them](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Quantifiers/Concept-Quantifiers.ipynb)&nbsp;&rarr;

---

## 3. Tests

**DeMorgan**, both forms, exhaustively.

In [ ]:
assert equiv2(lambda a, b: not (a and b), lambda a, b: (not a) or (not b))
assert equiv2(lambda a, b: not (a or b),  lambda a, b: (not a) and (not b))
print("%-8s %-8s %-14s %-14s" % ("a", "b", "!(a and b)", "!a or !b"))
for a, b in BOOLS:
    print("%-8s %-8s %-14s %-14s" % (a, b, not (a and b), (not a) or (not b)))
print("\nidentical columns -- and likewise for the dual")

**Contrapositive** is an equivalence, not a one-way rule.

In [ ]:
assert equiv2(IMP, lambda a, b: IMP(not b, not a))
print("%-8s %-8s %-12s %-12s" % ("a", "b", "a => b", "!b => !a"))
for a, b in BOOLS:
    print("%-8s %-8s %-12s %-12s" % (a, b, IMP(a, b), IMP(not b, not a)))

**The near-misses:** converse and inverse are NOT equivalent to the original.

In [ ]:
conv = lambda a, b: IMP(b, a)
inv  = lambda a, b: IMP(not a, not b)
print("  a=>b equivalent to its CONVERSE b=>a ?  ", equiv2(IMP, conv))
print("  a=>b equivalent to its INVERSE !a=>!b ? ", equiv2(IMP, inv))
assert not equiv2(IMP, conv) and not equiv2(IMP, inv)
bad = [(a, b) for a, b in BOOLS if IMP(a, b) != conv(a, b)]
print("  witness where a=>b differs from b=>a :", bad)
print("\n(converse and inverse ARE equivalent to each other -- each is the")
print(" contrapositive of the other.)")
assert equiv2(conv, inv)

The **set** version of DeMorgan.

In [ ]:
U = set(range(10))
A, B = {1, 2, 3, 4}, {3, 4, 5, 6}
d1, d2 = set_demorgan(U, A, B)
print("  complement(A n B) == comp(A) u comp(B) ?", d1)
print("  complement(A u B) == comp(A) n comp(B) ?", d2)
assert d1 and d2

And the **language** version, verified on real DFA.

In [ ]:
even0 = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')
even1 = md2mc('''DFA
IF : 1 -> Od
IF : 0 -> IF
Od : 1 -> IF
Od : 0 -> Od
''')
lhs = min_dfa(intersect_dfa(even0, even1))
rhs = min_dfa(comp_dfa(union_dfa(comp_dfa(even0), comp_dfa(even1))))
print("  L1 n L2 == complement(comp(L1) u comp(L2)) ?", langeq_dfa(lhs, rhs))
assert langeq_dfa(lhs, rhs) and iso_dfa(lhs, rhs)
print("\nSame law, three settings: propositions, sets, languages.")

## 4. Exercises


1. State DeMorgan for three operands. Does it generalise to $n$?
2. Give an English sentence whose converse sounds true but is not.
3. Which chapter-4 proof is licensed by the contrapositive equivalence?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Basics/Concept-DeMorgan-And-Contrapositive')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')